<center><h2>Академия Аналитиков Авито 2024-2025<br><br>Домашнее задание по Audio Processing</h2></center>

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

if "src" in os.getcwd():
    os.chdir("../")

import torch
import torchaudio
import jiwer
from IPython.display import display, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration

In [1]:
!pip install jiwer

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 10.2 MB/s eta 0:00:00


#### 🛠️ Утилиты для домашки

↓ Скачивание набора данных RIR-ов и шумов. Можно использовать другой набор данных.

In [3]:
import numpy as np

↓ Утилиты работы с обученной моделью

In [4]:
def postprocess_text(text: str) -> str:
    """Постобработка строк. 
    Приводим в нижний регистр, удаляем краевые пробельные 
    и неалфавитные символы (цифры, пунктуацию).
    """

    text = text.lower().strip()
    text = "".join(s for s in text if str(s).isalpha() or s == " ")

    return text


def infer_whisper(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    wav: torch.Tensor, 
    sample_rate: int, 
    lang: str = "en",
    device: str = "cuda:0"
) -> str:
    """Обработка Whisper-ом входного аудио."""

    # Обрабатываем аудио
    inputs = processor(
        wav.squeeze().numpy(),
        sampling_rate=sample_rate,
        return_tensors="pt",
    )

    # Генерируем транскрипцию
    outputs = model.generate(
        **inputs.to(device),
        language=f"<|{lang}|>"
    )

    hypothesis = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    return hypothesis

↓ Подсчёт Keyword Error Rate

In [5]:
def compute_ker(reference: str, hypothesis: str, keyword: str) -> float:
    reference = reference.lower()
    hypothesis = hypothesis.lower()
    keyword = keyword.lower()

    ref_keywords = [word for word in reference.split() if word == keyword]
    hyp_keywords = [word for word in hypothesis.split() if word == keyword]

    ref_keywords = " ".join(ref_keywords)
    hyp_keywords = " ".join(hyp_keywords)

    wer_transforms = jiwer.transforms.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(),
        jiwer.Strip(),
        jiwer.ReduceToListOfListOfWords()
    ])

    if len(ref_keywords) == 0:
        raise ValueError("There is no keywords in reference.")
    
    ker = jiwer.wer(
        ref_keywords, 
        hyp_keywords, 
        reference_transform=wer_transforms, 
        hypothesis_transform=wer_transforms
    )

    return ker

### **Формулировка задачи**

#### 💢 Проблема

Слово "Авито" распознаётся плохо оригинальным Whisper-ом. Либо он пишет его на латинице, либо не в той форме, либо путает с другими словами.

Пример:

In [12]:
device = "cuda:0"

# Инициализируем модель
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model.to(device)
processor = WhisperProcessor.from_pretrained("openai/whisper-small")

# Читаем аудио
wav, sr = torchaudio.load("./data/audio/hello_avito.wav")
display(Audio(data=wav, rate=sr))

# Рэсемплим в 16кГц (Whisper по-другому не умеет)
wav = torchaudio.transforms.Resample(sr, 16000)(wav)

# Запускаем модель на аудио
hypothesis = infer_whisper(
    model=model,
    processor=processor,
    wav=wav, 
    sample_rate=16000, 
    lang="ru",
    device=device
)
postprocess_text(hypothesis)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

'вас приветствует компания avita'

In [1]:
import torch
torch.cuda.is_available()

True

In [17]:
import asyncio
import edge_tts
import os
import json
import random

# Список фраз (всегда используем "Авито" в нужной форме для обучения)
phrases = [
    "Я нашел отличную машину на Авито",
    "Я разместил лбъявление на Авито",
    "Продай это через Авито, так быстрее",
    "На Авито сейчас много интересных вакансий",
    "Вчера купил на Авито новый телефон",
    "Авито помогает экономить деньги",
    "Посмотри это объявление в приложении Авито",
    "Свяжись со мной через сообщения на Авито",
    "Доставка Авито работает очень удобно",
    "Ты купил эту куртку на Авито?",
    "Сайт Авито очень быстро грузит",
    "Выставил старый ноутбук на Авито за пять минут",
    "Авито — это самая популярная доска объявлений"
]

voices = ["ru-RU-SvetlanaNeural", "ru-RU-DmitryNeural"]

async def generate_dataset(output_dir="./data/synthetic_avito", count=100):
    os.makedirs(output_dir, exist_ok=True)
    metadata = []

    for i in range(count):
        text = random.choice(phrases)
        voice = random.choice(voices)
        filename = f"avito_sample_{i}.mp3"
        filepath = os.path.join(output_dir, filename)
        
        # Генерация аудио
        communicate = edge_tts.Communicate(text, voice)
        await communicate.save(filepath)
        
        metadata.append({"audio_path": filepath, "sentence": text})
        if (i + 1) % 10 == 0:
            print(f"Сгенерировано {i + 1}/{count} файлов...")

    # Сохраняем разметку для обучения
    with open(os.path.join(output_dir, "metadata.jsonl"), "w", encoding="utf-8") as f:
        for entry in metadata:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")


await generate_dataset(count=100)

Сгенерировано 10/100 файлов...
Сгенерировано 20/100 файлов...
Сгенерировано 30/100 файлов...
Сгенерировано 40/100 файлов...
Сгенерировано 50/100 файлов...
Сгенерировано 60/100 файлов...
Сгенерировано 70/100 файлов...
Сгенерировано 80/100 файлов...
Сгенерировано 90/100 файлов...
Сгенерировано 100/100 файлов...


In [18]:
import torch
import torchaudio
import librosa
import numpy as np
from datasets import Dataset, Audio
from transformers import (
    WhisperForConditionalGeneration, 
    WhisperProcessor, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union

MODEL_ID = "openai/whisper-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = "./whisper-small-avito-finetuned_v2"

texts = [
    "разместил объявление на Авито",
    "посмотри это на Авито",
    "купил телефон через Авито",
    "Авито это удобно",
    "нашел квартиру на Авито"
]

def create_large_dataset(metadata_path="./data/synthetic_avito/metadata.jsonl"):
    data = {"audio_path": [], "sentence": []}
    with open(metadata_path, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)
            data["audio_path"].append(entry["audio_path"])
            data["sentence"].append(entry["sentence"])
    return Dataset.from_dict(data)

def prepare_dataset(batch):
    audio_path = batch["audio_path"]
    wav, _ = librosa.load(audio_path, sr=16000)
    
    wav_8k = librosa.resample(wav, orig_sr=16000, target_sr=8000)
    wav_final = librosa.resample(wav_8k, orig_sr=8000, target_sr=16000)

    batch["input_features"] = processor.feature_extractor(
        wav_final, sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Russian", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

new_tokens = [" Авито", "Авито"]
num_added_toks = processor.tokenizer.add_tokens(new_tokens)
model.resize_token_embeddings(len(processor.tokenizer)) 
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="russian", task="transcribe")
model.generation_config.suppress_tokens = []
model.generation_config.language = "russian"
model.generation_config.task = "transcribe"
for key in ["forced_decoder_ids", "suppress_tokens"]:
    if hasattr(model.config, key):
        delattr(model.config, key)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

raw_dataset = create_large_dataset()
vectorized_dataset = raw_dataset.map(prepare_dataset, remove_columns=raw_dataset.column_names)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    fp16=True,
    eval_strategy="no", 
    save_steps=500,
    predict_with_generate=True,
    generation_max_length=225,
    push_to_hub=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=vectorized_dataset,
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    processing_class=processor.feature_extractor,
)
model.freeze_encoder() 
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

C:\Users\User\anaconda3\Lib\site-packages\accelerate\utils\memory.py:46: RuntimeWarning: coroutine 'generate_dataset' was never awaited
  gc.collect()


Step,Training Loss
500,0.942544
1000,0.000209


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['./whisper-small-avito-finetuned_v2\\processor_config.json']

In [19]:
MODEL_PATH = "./whisper-small-avito-finetuned"
device = "cuda:0"
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)

def transcribe(wav: torch.Tensor, sr: int) -> str:
    wav = torchaudio.transforms.Resample(sr, 16000)(wav) 
    hypothesis = infer_whisper(
    model=model,
    processor=processor,
    wav=wav, 
    sample_rate=16000, 
    lang="ru",
    device=device
    )
    return postprocess_text(hypothesis)
    
wav, sr = torchaudio.load("./data/audio/hello_avito.wav")
transcribe(wav, sr)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

'привет авито'

In [20]:
device = "cuda:0"
MODEL_PATH = "./whisper-small-avito-finetuned"
# Инициализируем модель
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)

# Читаем аудио
wav, sr = torchaudio.load("./data/audio/hello_avito.wav")

# Рэсемплим в 16кГц (Whisper по-другому не умеет)
wav = torchaudio.transforms.Resample(sr, 16000)(wav)

# Запускаем модель на аудио
hypothesis = infer_whisper(
    model=model,
    processor=processor,
    wav=wav, 
    sample_rate=16000, 
    lang="ru",
    device=device
)
postprocess_text(hypothesis)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

'привет авито'

#### 😼 Что нужно сделать?

<span style="color: #cc6104;">От вас требуется заставить модель `openai/whisper-small` говорить слово "авито" в надлежащей форме (именительный падеж, единственное число).</span>

Можно пользоваться любыми способами, кроме очевидных:
1. Делать постпроцессинг распознанной строки, применяя регулярные выражения (или еще что-нибудь).
2. Накладывать ограничения на логиты во время генерации текста моделью (constrained decoding).

Любое другое соображение предлагаю обсуждать в личных сообщениях со мной, чтобы я давал добро или накладывал на него вето.

В общем случае ваша модель должна сама, "от природы", генерировать правильную форму слова "авито" там, где оно было произнесено.

#### 🧑‍🏫️ Описание прикладной области

Система будет работать с 8 кГц моноканальным аудио русской речи, записанной с микрофона телефона во время диалога. На каждой записи находится только один диктор -- другой собеседник ему не мешает. Диктор находится близко к микрофону, речь разборчива (как в примере выше). Часто диктор идёт по улице, едет в транспорте или сидит где-нибудь в столовой. Для экономии места записи были сконвертированы в mp3-формат.

#### 🤖 Формат решения

Ожидаю архив с двумя вещами:
1. Чекпоинт модели: её веса + конфиги processor-а. В общем случае, это папка, которая создается в процессе или результате обучения модели.
2. Python-скрипт с функцией `transcribe`, принимающей на вход одну пару "аудио, частота сэмплирования" и возвращающей одну строку -- транскрипцию.

#### 🥇 По какой метрике я буду вести оценку качества?

[Keyword Error Rate (KER)](https://www.isca-archive.org/interspeech_2008/park08b_interspeech.pdf):

$$KER = \frac{N_{miss} + N_{detect}}{L_{ref}}$$

где 
- $N_{miss}$ -- количество ошибочно упущенных ключевых слов; 
- $N_{detect}$ -- количество ошибочно распознанных ключевых слов; 
- $L_{ref}$ -- количество ключевых слов в референсе.

Имплементация приведена в разделе "Утилиты".